<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Deep Learning for MNIST Classification — Algorithm</b></h1>
</div>


## Purpose and Reading Convention

This notebook explains how to implement and review this lab inside its existing `labs/` directory. The executable code remains in [main.ipynb](main.ipynb); no repository split or Python package extraction is required.

Each numbered step gives **Purpose**, **Inputs**, **Procedure**, **Outputs**, **Validation** and **On failure**. Follow dependencies in order and reuse the same intermediate arrays for metrics and figures.

**Validation convention:** these are review contracts. Some are enforced by existing assertions or exceptions; others require inspecting intermediate results. A documentation update does not add runtime guards. The *Implementation limits* section identifies important gaps so the algorithm does not claim unimplemented protections.

Use parameter values and numeric tolerances actually declared in `main.ipynb`. Do not replace them with unsupported universal thresholds. A failed implementation check stops the run; a diagnostic limitation must be reported rather than silently treated as a pass.

## Inputs and Final Outputs

| Item | Description |
| --- | --- |
| **Inputs** | MNIST IDX training/testing files |
| **Final outputs** | Three trained MLP variants, training histories, test metrics, prediction diagnostics, correctness-retention confidence curves |
| **Implementation target** | `main.ipynb` |

## 0. Start

**Purpose**

Establish the scope and a clean execution order.

**Inputs**

The problem statement, theory, requirements and main notebook in this lab.

**Procedure**

1. Select the project `.venv` kernel in VS Code.
2. Use this notebook only as the implementation recipe for `main.ipynb`.
3. Keep training, testing, and confidence analysis separate so evaluation remains out-of-sample.

**Outputs**

Selected environment and a top-to-bottom execution plan.

**Validation**

Review the local assumptions before using cached notebook outputs.

**On failure**

Resolve missing prerequisites first; do not infer a successful run from stored outputs.

## 1. Create the Reproducible Setup

**Purpose**

Make architecture comparisons repeatable.

**Inputs**

PyTorch environment and declared hyperparameters.

**Procedure**

1. Import `Path`, NumPy, Matplotlib, PyTorch, `python-mnist`, `Dataset`, `DataLoader`, and `tqdm`.
2. Set one fixed random seed in NumPy and PyTorch.
3. Select CUDA when available, otherwise CPU.
4. Define batch size, hidden widths `[128,256,512]`, epochs, learning rate, image size, class count, and flattened input dimension.

**Outputs**

Seed 42; device; batch 512; 10 epochs; widths 128/256/512.

**Validation**

Use learning rate 1e-3 and weight decay 1e-3 consistently; a seed alone does not guarantee identical GPU runs.

**On failure**

Resolve dependency/device failures before training.

## 2. Validate MNIST Files and Output Paths

**Purpose**

Use the intended local MNIST split.

**Inputs**

DATA_DIR and four exact IDX filenames.

**Procedure**

1. Point `DATA_DIR` to the local MNIST folder and create `OUTPUT_DIR`.
2. Discover the four required IDX files by exact filename.
3. Stop if any file is missing.
4. Keep all generated figures inside the lab output directory.

**Outputs**

Verified file inventory; figure directory.

**Validation**

Both image and label files exist for training and testing.

**On failure**

Raise the existing missing-file error rather than replacing a split silently.

## 3. Implement the MNIST Dataset

**Purpose**

Provide one consistent sample representation.

**Inputs**

Training or testing IDX partition.

**Procedure**

1. Create a `Dataset` subclass receiving `training` or `testing` plus the MNIST directory.
2. Load the requested IDX partition.
3. Store images and labels and return partition length in `__len__`.
4. In `__getitem__`, convert one image to a normalized float32 vector and return it with its integer label.

**Outputs**

MNISTDataset; image vectors (784,) float32 and integer labels.

**Validation**

Pixels divided by 255; labels from 0 to 9; sample counts match labels.

**On failure**

Reject unsupported partition names; inspect corrupt or mispaired IDX data.

## 4. Build DataLoaders

**Purpose**

Feed mini-batches without mixing evaluation and training.

**Inputs**

MNISTDataset and batch size.

**Procedure**

1. Create a helper that wraps a dataset in a DataLoader.
2. Shuffle training but keep testing deterministic.
3. Create training/testing datasets and loaders with the same batch size.
4. Verify exactly 60,000 training samples and 10,000 test samples.

**Outputs**

Training/testing DataLoaders.

**Validation**

60,000 training and 10,000 testing samples; only training is shuffled.

**On failure**

Stop on split-size or batching errors.

## 5. Inspect Representative Samples

**Purpose**

Check that input preprocessing preserves digit meaning.

**Inputs**

Deterministic training sample indices.

**Procedure**

1. Select a deterministic subset of training indices.
2. Reshape flattened vectors to 28×28 only for visualization.
3. Plot images with labels and save the diagnostic figure.
4. Use this to catch wrong normalization, shapes, or labels before training.

**Outputs**

28 by 28 sample grid with labels.

**Validation**

Displayed image and label refer to the same sample; flattening is reversed only for display.

**On failure**

Correct ordering or normalization before fitting models.

## 6. Define the MLP Classifier

**Purpose**

Define a controlled width-only comparison.

**Inputs**

784 features, hidden width, 10 classes.

**Procedure**

1. Create a module parameterized by hidden-layer width.
2. Build `784 → hidden_width → 10` linear layers.
3. Insert BatchNorm and ReLU between linear layers.
4. Use cross-entropy directly on logits during training.
5. During inference apply softmax only to decode class and confidence.

**Outputs**

MNISTClassifier with Linear, BatchNorm, ReLU and Linear.

**Validation**

Output logits have 10 columns; cross-entropy receives logits, not softmax probabilities.

**On failure**

Correct layer dimensions or loss inputs before training.

## 7. Verify One Forward Pass

**Purpose**

Catch model/batch incompatibility cheaply.

**Inputs**

One training batch and a model on the selected device.

**Procedure**

1. Instantiate one representative model and move it to the selected device.
2. Read one training batch and move tensors to the same device.
3. Run one forward pass in training mode.
4. Verify finite scalar loss and logits shaped `(batch,10)` before launching full training.

**Outputs**

Scalar loss and logits shaped (batch, 10).

**Validation**

Finite loss/logits; labels are int64; batch and model share the device.

**On failure**

Stop before the epoch loop if shapes, dtype or device disagree.

## 8. Implement Batch Preparation

**Purpose**

Share batch conversion across training and inference.

**Inputs**

NumPy image and label batches.

**Procedure**

1. Create one helper converting image and label batches to tensors on the requested device.
2. Preserve image float type and integer class labels.
3. Use this helper in both training and evaluation.

**Outputs**

Device tensors under image and label keys.

**Validation**

Images remain float32 and labels int64; batch lengths agree.

**On failure**

Correct data preparation instead of casting predictions to repair errors.

## 9. Implement the Training Loop

**Purpose**

Update weights and measure sample-weighted loss.

**Inputs**

Training loader, model, Adam optimizer, epoch count.

**Procedure**

1. For each epoch set `model.train()`.
2. For each batch: prepare tensors, zero gradients, compute loss, backpropagate, and step the optimizer.
3. Accumulate loss weighted by batch size.
4. Store one mean training loss per epoch and return the full history.

**Outputs**

epoch_loss_history with one value per epoch.

**Validation**

Zero gradients each batch; backpropagate only training loss; divide weighted loss by processed sample count.

**On failure**

Stop for nonfinite loss; investigate optimization settings rather than fabricate a decreasing history.

## 10. Implement the Evaluation Loop

**Purpose**

Measure predictions without changing model statistics.

**Inputs**

Trained model and testing loader.

**Procedure**

1. Use `torch.no_grad()` and call `model.eval()`.
2. For each test batch collect predicted classes, confidences, labels, and logits on CPU.
3. Concatenate batch outputs.
4. Compute test accuracy and return all evaluation arrays plus accuracy.

**Outputs**

cls, prob, label, logits, accuracy.

**Validation**

eval mode and no_grad; 10,000 aligned predictions; confidence in [0,1].

**On failure**

Restore eval mode or alignment before interpreting accuracy.

## 11. Train the Three Width Variants

**Purpose**

Run the same experiment for all three widths.

**Inputs**

Widths, seed, common training parameters.

**Procedure**

1. Create dictionaries for models, histories, and evaluation results.
2. For each hidden width reset the PyTorch seed.
3. Instantiate the model and Adam optimizer, train for all epochs, then evaluate on the test loader.
4. Store model, loss history, and test result under the width key.

**Outputs**

models, training_histories, model_evaluation_results.

**Validation**

Every width has 10 finite epoch losses and one held-out evaluation.

**On failure**

Report an interrupted run; do not compare an incomplete history to a full one.

## 12. Compare Learning Behaviour

**Purpose**

Compare measured optimization behaviour.

**Inputs**

Three histories and test evaluations.

**Procedure**

1. Plot training loss versus epoch for all widths.
2. Print each final training loss and test accuracy.
3. Keep reference reported losses separate from new measurements.
4. Follow the notebook's declared selection rule rather than changing it after seeing test results.

**Outputs**

Loss curves and reported test accuracies.

**Validation**

Keep reference values separate from this run; selection uses lowest final training loss.

**On failure**

Do not select by test accuracy or present historical values as new measurements.

## 13. Visualize Predictions

**Purpose**

Inspect predictions and their confidence distribution.

**Inputs**

Each trained model and the first 15 test examples.

**Procedure**

1. Choose a fixed subset of test images.
2. For each model run inference and display predicted class/confidence against the true label.
3. Keep `model.eval()` active.
4. Save the prediction diagnostics.

**Outputs**

Prediction grids and class-probability bars.

**Validation**

Same examples across widths; probabilities sum approximately to 1 per sample.

**On failure**

Investigate confidence/class mismatches before plotting conclusions.

## 14. Compute Confidence-Threshold Curves

**Purpose**

Describe exactly what confidence filtering measures.

**Inputs**

Aligned predicted classes, confidence and labels.

**Procedure**

1. Read predicted classes, confidence and labels from the same stored evaluation result; compute the boolean `correct` array.
2. Generate 100 thresholds between 0 and 0.99 and use the strict rule `confidence > threshold`.
3. Count accepted-correct (`TP`), accepted-incorrect (`FP`) and rejected-correct (`FN`) examples.
4. Compute `precision=TP/(TP+FP)`, `recall=TP/(TP+FN)` and the existing `accepted_accuracy=TP/N`.
5. Return NaN precision if none are accepted and zero recall if no predictions are correct; reorder all arrays consistently by recall.
6. Coverage would be `(TP+FP)/N`, but it is not returned by this implementation. Precision here is accuracy among accepted predictions; `accepted_accuracy` uses all test samples as its denominator.

**Outputs**

Thresholds, precision, recall, accepted_accuracy arrays.

**Validation**

accepted means confidence>threshold; precision=accepted_correct/accepted; recall=accepted_correct/all_correct; accepted_accuracy=accepted_correct/N.

**On failure**

Precision is NaN when none are accepted; retain this undefined case. Coverage is not returned by the current helper.

## 15. Analyze the Selected Model

**Purpose**

Apply the declared selection rule to the confidence illustration.

**Inputs**

Final training losses and stored evaluations.

**Procedure**

1. Select `best_hidden_size` by the smallest final value in `training_histories`, as implemented.
2. Use that width's stored test result to compute confidence curves.
3. Save the precision–recall and accepted-correct/all-test-samples–recall plots.
4. Explain that these are correctness-retention curves, not per-digit precision–recall curves or evidence that confidence is calibrated.

**Outputs**

best_hidden_size and confidence curves.

**Validation**

argmin final training loss determines width; all curve arrays are reordered together by recall.

**On failure**

Do not call this standard per-class precision/recall or claim the selected width is independently validated as optimal.

## 16. Run Final Validation

**Purpose**

Verify the completed experiment and saved figures.

**Inputs**

Datasets, three model results, histories and required files.

**Procedure**

1. Recheck dataset sizes and confirm all three widths were trained.
2. Verify every loss history has expected length and finite values.
3. Verify predictions/confidences/labels have matching lengths and valid ranges.
4. Verify every declared output exists before printing the final pass message.

**Outputs**

Validation outcome and six figures.

**Validation**

Expected sample counts; complete finite histories; matching prediction arrays; valid confidence ranges.

**On failure**

Fail on missing models, invalid arrays or missing plots; undefined threshold precision is distinct from invalid model confidence.

## Complete End-to-End Pseudocode

This is control-flow pseudocode, not an additional Python API. It follows the current implementation; review requirements and known gaps are distinguished below.

```text
set NumPy/PyTorch seed, device and shared hyperparameters
require all four local IDX files
build MNISTDataset training/testing partitions and their DataLoaders
check 60000/10000 samples; inspect a fixed training sample grid
define MNISTClassifier: Linear(784,width), BatchNorm, ReLU, Linear(width,10)
verify one training-mode forward pass produces finite loss and (batch,10) logits
for width in [128,256,512]:
    reset torch seed; create fresh model and Adam optimizer
    for epoch in range(EPOCHS):
        set train mode; reset weighted loss and sample count
        for training batch:
            prepare tensors; zero gradients; forward; backward; optimizer step
            accumulate loss * batch_size and sample count
        append weighted_loss / sample_count
    set eval mode; disable gradients
    collect classes, confidence, labels and logits over every test batch
    store accuracy, trained model and history under width
plot histories; print training loss and test accuracy separately
plot predictions for the same first 15 test samples for every width
width <- argmin final training loss, not argmax test accuracy
correct <- predicted_class == label for the selected model
for threshold in linspace(0,0.99,100):
    accepted <- confidence > threshold
    TP <- count(accepted AND correct)
    FP <- count(accepted AND NOT correct)
    FN <- count(NOT accepted AND correct)
    precision <- TP/(TP+FP), or NaN when none accepted
    recall <- TP/(TP+FN), or zero when no correct predictions
    accepted_accuracy <- TP / total_test_samples
reorder all curve arrays by recall and plot them
validate datasets, histories, evaluation arrays and required files
```

## Task Coverage

Every task defined in the Problem Statement is covered by the coding sequence above:

- **Task 1:** Validate MNIST Data and Output Paths
- **Task 2:** Build the MNIST Dataset and Mini-Batch Pipeline
- **Task 3:** Load and Validate Training and Testing Data
- **Task 4:** Visualize Representative MNIST Samples
- **Task 5:** Define the One-Hidden-Layer MLP Classifier
- **Task 6:** Verify the Model Architecture and Forward Pass
- **Task 7:** Define Training and Evaluation Utilities
- **Task 8:** Train the 128-, 256-, and 512-Neuron Models
- **Task 9:** Compare Training Loss and Test Accuracy
- **Task 10:** Visualize Predictions and Confidence Scores
- **Task 11:** Compute Confidence-Threshold Precision, Recall, and Accepted Accuracy
- **Task 12:** Analyze the Best Model and Save Evaluation Curves
- **Task 13:** Run Numerical and Output-file Validation Checks


## Notebook Relationship

- `problem_statement.ipynb` defines **what must be solved**.
- `theory.ipynb` explains **why the methods work**.
- `requirements.ipynb` defines **what is needed to run the lab**.
- `algorithm.ipynb` defines **how to build the solution step by step**.
- `main.ipynb` is the **executable implementation** of this algorithm.


## Required Output Contract

The current `REQUIRED_OUTPUTS` list in `main.ipynb` contains 6 files under `../outputs/figures/`:

- `mnist_sample_batch.png`
- `training_loss_comparison.png`
- `mnist_predictions_128.png`
- `mnist_predictions_256.png`
- `mnist_predictions_512.png`
- `pr_accuracy_curve.png`

A presence check detects missing artifacts, not stale artifacts. Inspect the figures from the run being reported.

## Implementation Traceability

Reviewed against [source revision 66078c2](https://github.com/denoskume/msc-coro-dassip-portfolio/tree/66078c2481115d0b2596c6926e0750d136ba1005/labs/computer-vision/deep-learning/notebooks).

The following helpers/classes already exist in `main.ipynb`; algorithm operations that are not listed here are inline notebook cells or descriptive pseudocode:

- `MNISTDataset`
- `build_mnist_dataset_and_loader`
- `MNISTClassifier`
- `prepare_training_batch`
- `train_classifier`
- `evaluate_classifier`
- `plot_classifier_predictions`
- `compute_confidence_threshold_curves`

## Implementation Limits and Interpretation

- The selected width minimizes final **training loss** for a confidence-curve demonstration. It is not selected by validation-set generalization, and the test set must not be used to tune thresholds while claiming an independent final score.
- `precision` measures correctness among accepted predictions. `recall` measures retained correct predictions among all correct predictions. The variable `accepted_accuracy` is `accepted_correct / all_test_samples`. Coverage is a different quantity and is not returned.
- NaN precision for an empty accepted set is intentional. A high softmax confidence is not evidence of probability calibration.
- Seeds and common settings improve repeatability; they do not guarantee bitwise identity across hardware or prove that accuracy will exceed a fixed target.

## 17. End

The coding plan ends after the implemented checks and required artifacts have been reviewed. Interpret outputs beside their corresponding experiment, then write the overall analysis at the bottom of `main.ipynb`. Report the implementation limits below; a numerical pass alone is not proof of physical validity or application-level accuracy.